# buffer-copy_-inplace — ex2: predict copy_ behavior across dtype + shape combinations

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `buffer-copy_-inplace`. Running the final beacon cell reports progress against the `PyTorch: in-place buffer copy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: in-place buffer copy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`buffer-copy_-inplace`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "buffer-copy_-inplace"
DD_SUBTOPIC = "PyTorch: in-place buffer copy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `tensor.copy_(other)` — quick refresher

`dst.copy_(src)` is the canonical in-place "overwrite my contents with src's contents" op. It:

- preserves `dst.data_ptr()` (no reallocation),
- preserves `dst.dtype` (casts `src` to `dst.dtype` if they differ),
- **broadcasts** `src` against `dst.shape` (so a scalar into a vector works),
- requires `src` to be broadcastable to `dst` — a `(3,)` src into a `(4,)` dst RAISES.

**What it is NOT.** It is not `dst = src` (rebinds the Python name) and not `dst.data = src` (swaps storage, breaks registered-buffer links). For module buffers / parameters, `copy_` is the only safe in-place overwrite.

### Exercise 2 — predict copy_ behavior across dtype + shape combinations

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the broadcasting and dtype-coercion rules of `tensor.copy_` to predict, for each (dst, src) pair, whether the call succeeds + what dtype/shape/storage the destination ends up with.
> Keywords: copy_, broadcasting, dtype-coercion, in-place-rules
> ```

**KCs targeted:** `buffer-copy_-inplace`, `copy_-broadcasts-and-coerces`

Implement `ex2_predict_copy_outcome(dst, src)`. You will NOT call `.copy_` — you will REASON about what it would do, and return a dict describing the predicted outcome:

Return `{'ok': bool, 'dtype': torch.dtype | None, 'shape': tuple | None}`:

- `'ok'` — True iff `dst.copy_(src)` would succeed (i.e. `src` is broadcastable to `dst.shape`). False otherwise.
- `'dtype'` — `dst.dtype` if ok (copy_ keeps dst dtype), else None.
- `'shape'` — `tuple(dst.shape)` if ok, else None.

**Broadcast rule:** `src` is broadcastable to `dst` iff, aligned right, every src dim is 1 or equal to the matching dst dim, and `src.dim() <= dst.dim()`.

After computing your prediction, the test will actually run `dst_copy.copy_(src)` (on a clone of dst) and assert your prediction matches reality — including the success/failure side.

In [ ]:
def ex2_predict_copy_outcome(dst: Tensor, src: Tensor) -> dict:
    """Predict the result of dst.copy_(src) without calling it."""
    raise NotImplementedError()


def _test_ex2():
    def _broadcastable(src_shape, dst_shape):
        # Reference oracle for broadcastability of src into dst.
        if len(src_shape) > len(dst_shape):
            return False
        for s, d in zip(reversed(src_shape), reversed(dst_shape)):
            if s != 1 and s != d:
                return False
        return True

    cases = [
        # (label, dst, src)
        ('same-shape-same-dtype',  t.zeros(3, 4, dtype=t.float32),
                                    t.ones(3, 4, dtype=t.float32)),
        ('same-shape-cross-dtype', t.zeros(3, 4, dtype=t.float32),
                                    t.ones(3, 4, dtype=t.float64)),
        ('scalar-into-vector',     t.zeros(5, dtype=t.float32),
                                    t.tensor(7.0, dtype=t.float32)),
        ('row-into-matrix',        t.zeros(3, 4, dtype=t.float32),
                                    t.ones(4, dtype=t.float32)),
        ('mismatched-vector',      t.zeros(4, dtype=t.float32),
                                    t.ones(3, dtype=t.float32)),
        ('extra-leading-dim',      t.zeros(4, dtype=t.float32),
                                    t.ones(1, 4, dtype=t.float32)),
        ('int-into-float',         t.zeros(2, 3, dtype=t.float32),
                                    t.ones(2, 3, dtype=t.int64)),
    ]

    for label, dst, src in cases:
        pred = ex2_predict_copy_outcome(dst, src)
        assert set(pred.keys()) == {'ok', 'dtype', 'shape'}, (
            f'{label}: dict must have exactly keys ok/dtype/shape, got {pred.keys()}'
        )
        expected_ok = _broadcastable(tuple(src.shape), tuple(dst.shape))
        assert pred['ok'] == expected_ok, (
            f'{label}: ok prediction wrong — got {pred["ok"]}, expected {expected_ok}'
        )
        if expected_ok:
            assert pred['dtype'] == dst.dtype, (
                f'{label}: dtype should equal dst.dtype={dst.dtype}, got {pred["dtype"]}'
            )
            assert pred['shape'] == tuple(dst.shape), (
                f'{label}: shape should equal {tuple(dst.shape)}, got {pred["shape"]}'
            )
        else:
            assert pred['dtype'] is None and pred['shape'] is None, (
                f'{label}: on failure dtype + shape must both be None'
            )

        # Cross-check against reality.
        dst_copy = dst.clone()
        try:
            dst_copy.copy_(src)
            real_ok = True
        except RuntimeError:
            real_ok = False
        assert real_ok == expected_ok, (
            f'{label}: oracle/reality mismatch — fix the test'
        )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_predict_copy_outcome(dst: Tensor, src: Tensor) -> dict:
    src_shape = tuple(src.shape)
    dst_shape = tuple(dst.shape)
    if len(src_shape) > len(dst_shape):
        return {'ok': False, 'dtype': None, 'shape': None}
    for s, d in zip(reversed(src_shape), reversed(dst_shape)):
        if s != 1 and s != d:
            return {'ok': False, 'dtype': None, 'shape': None}
    return {'ok': True, 'dtype': dst.dtype, 'shape': dst_shape}
```

**Two-rule broadcast.** Aligned right, every src dim must be 1 or equal to the matching dst dim, AND `src.dim() <= dst.dot.dim()`. The second rule is easy to forget — `(1, 4).copy_into((4,))` fails because src has more dims than dst.

**Dtype is dst-driven.** Unlike `t.add` (which promotes), `copy_` keeps dst's dtype and silently casts src — `int64 → float32` is legal and lossless for small ints. `float64 → float32` is also legal but loses precision, no warning.

**Why not just call copy_.** This is an Analyze drill — the value is internalizing the rule so you can predict downstream behavior without trial-and-error in a Jupyter cell.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()